# Multivariate Time-Series Forecasting

## 2. RNN Model

This notebook develops a Recurrent Neural Network (RNN) for multivariate time-series demand forecasting.

The RNN will learn temporal patterns from historical observations and use them to predict future sales.

The model will use a sequence of historical observations as input and predict the next day's demand.

The same time-based validation framework used in the baseline stage will be maintained to ensure a fair comparison.

The RNN performance will be evaluated using:

- MAE
- RMSE
- WAPE

The RNN will be compared against the baseline models established in the previous stage.

In [1]:
import gc
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping

print("Libraries imported successfully.")
print("TensorFlow version:", tf.__version__)

Libraries imported successfully.
TensorFlow version: 2.20.0


In [2]:
print("Available GPUs:")
print(tf.config.list_physical_devices("GPU"))

Available GPUs:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# RNN forecasting configuration

SEQUENCE_LENGTH = 28
FORECAST_HORIZON = 1

TARGET_COLUMN = "sales"

FEATURE_COLUMNS = [
    "sales",
    "sell_price",
    "price_available",
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7",
    "is_event_day",
    "event_count",
    "snap_active"
]

print("====================================")
print("RNN Configuration")
print("====================================")

print("Sequence length:", SEQUENCE_LENGTH)
print("Forecast horizon:", FORECAST_HORIZON)
print("Number of input features:", len(FEATURE_COLUMNS))
print("Target:", TARGET_COLUMN)

RNN Configuration
Sequence length: 28
Forecast horizon: 1
Number of input features: 22
Target: sales


In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
import os

print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))

if os.path.exists("/content/drive/MyDrive"):
    print("\nMyDrive contents:")
    print(os.listdir("/content/drive/MyDrive")[:50])

MyDrive exists: True

MyDrive contents:
['20240322_184337.jpg', 'Sanjay Arora Bansi Lal.pdf', 'Emailing Jam Ma Real analysis pyq.pdf', 'tifr Linear Algebra Pyq.pdf', 'Jam Ma Real analysis pyq.pdf', 'NBHM Linear Algebra Pyq.pdf', 'Csir net Linear Algebra Pyq.pdf', 'Gate Ma Linear Algebra pyq.pdf', 'Jam Ma Linear Algebra Pyq.pdf', 'EWS form (1).pdf', 'Screenshot_20240322-181320_Chrome.jpg', 'M121M89AdmitCard (1) (2) (1).pdf', '20240322_190639.jpg', 'AUD-20231007-WA0038.m4a', 'AUD-20231010-WA0060.mp3', 'AUD-20231010-WA0075.mp3', 'AUD-20231011-WA0086.mp3', 'gitika 1.mp3', 'Call recording +917023252073_221029_181115.m4a', 'Call recording +917023252073_221029_181455.m4a', 'Call recording +918955517480_230827_110831.m4a', 'Call recording +918955517480_230827_110839.m4a', 'Call recording +918955517480_230827_110949.m4a', 'Call recording +918955517480_230827_111305.m4a', 'Call recording 9513392208_240331_141850.m4a', 'Call recording amita jaipur_230423_132825.m4a', 'Call recording amita jaipur_

In [8]:
# Load the final feature dataset

data_path = "/content/drive/MyDrive/features_event_snap.parquet"

parquet_file = pq.ParquetFile(data_path)

print("Dataset loaded successfully.")

print("Rows:", f"{parquet_file.metadata.num_rows:,}")
print("Columns:", len(parquet_file.schema.names))
print("Row groups:", parquet_file.num_row_groups)

Dataset loaded successfully.
Rows: 58,327,370
Columns: 42
Row groups: 61


In [9]:
# Test sequence creation on a single item-store series

sequence_test_columns = [
    "item_id",
    "store_id",
    "date"
] + FEATURE_COLUMNS

# Read only the first row group for testing
test_df = parquet_file.read_row_group(
    0,
    columns=sequence_test_columns
).to_pandas()

test_df["date"] = pd.to_datetime(test_df["date"])

# Select one item-store series
test_item = test_df["item_id"].iloc[0]
test_store = test_df["store_id"].iloc[0]

test_series = (
    test_df[
        (test_df["item_id"] == test_item) &
        (test_df["store_id"] == test_store)
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

print("====================================")
print("Sequence Test")
print("====================================")

print("\nItem:", test_item)
print("Store:", test_store)

print("\nSeries rows:", len(test_series))

print("\nDate range:")
print(test_series["date"].min(), "→", test_series["date"].max())

print("\nExpected sequence input shape:")
print(f"({SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})")

del test_df
gc.collect()

Sequence Test

Item: HOBBIES_1_001
Store: CA_1

Series rows: 1049

Date range:
2011-01-29 00:00:00 → 2013-12-12 00:00:00

Expected sequence input shape:
(28, 22)


0

In [10]:
# Check missing values in RNN input features
# Memory-efficient: process one Parquet row group at a time

missing_counts = pd.Series(0, index=FEATURE_COLUMNS, dtype="int64")
total_rows = 0

for rg in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(
        rg,
        columns=FEATURE_COLUMNS
    ).to_pandas()

    missing_counts += df.isna().sum()
    total_rows += len(df)

    del df
    gc.collect()

print("====================================")
print("Missing Value Check")
print("====================================")

print("Total rows checked:", f"{total_rows:,}")

print("\nMissing values by feature:")
print(missing_counts[missing_counts > 0])

print("\nTotal features with missing values:",
      (missing_counts > 0).sum())

Missing Value Check
Total rows checked: 58,327,370

Missing values by feature:
sell_price            12299413
lag_1                    30490
lag_7                   213430
lag_14                  426860
lag_28                  853720
rolling_mean_7          213430
rolling_mean_28         853720
rolling_std_7           213430
rolling_std_28          853720
price_change_1        12299413
price_change_pct_1    12299413
price_relative_7      12512843
dtype: int64

Total features with missing values: 12


In [11]:
# Check whether missing values occur only at the beginning of each series

check_columns = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28"
]

missing_position_check = {
    col: {
        "missing_rows": 0,
        "missing_after_first_28_days": 0
    }
    for col in check_columns
}

for rg in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(
        rg,
        columns=["item_id", "store_id", "date"] + check_columns
    ).to_pandas()

    df["date"] = pd.to_datetime(df["date"])

    for col in check_columns:
        missing = df[col].isna()

        missing_position_check[col]["missing_rows"] += int(missing.sum())

        # Approximate check:
        # missing values after the first 28 calendar days
        if missing.any():
            first_dates = (
                df.groupby(["item_id", "store_id"])["date"]
                .transform("min")
            )

            days_from_start = (df["date"] - first_dates).dt.days

            missing_position_check[col]["missing_after_first_28_days"] += int(
                (missing & (days_from_start >= 28)).sum()
            )

    del df
    gc.collect()

print("====================================")
print("Missing Position Check")
print("====================================")

for col, values in missing_position_check.items():
    print(f"\n{col}")
    print("Missing rows:", f"{values['missing_rows']:,}")
    print(
        "Missing after first 28 days:",
        f"{values['missing_after_first_28_days']:,}"
    )

Missing Position Check

lag_1
Missing rows: 30,490
Missing after first 28 days: 0

lag_7
Missing rows: 213,430
Missing after first 28 days: 0

lag_14
Missing rows: 426,860
Missing after first 28 days: 0

lag_28
Missing rows: 853,720
Missing after first 28 days: 0

rolling_mean_7
Missing rows: 213,430
Missing after first 28 days: 0

rolling_mean_28
Missing rows: 853,720
Missing after first 28 days: 0

rolling_std_7
Missing rows: 213,430
Missing after first 28 days: 0

rolling_std_28
Missing rows: 853,720
Missing after first 28 days: 0


In [12]:
# Test RNN input preparation on one row group

test_columns = [
    "item_id",
    "store_id",
    "date"
] + FEATURE_COLUMNS

test_df = parquet_file.read_row_group(
    0,
    columns=test_columns
).to_pandas()

test_df["date"] = pd.to_datetime(test_df["date"])

print("====================================")
print("RNN Input Preparation Test")
print("====================================")

print("Rows:", len(test_df))
print("Features:", len(FEATURE_COLUMNS))

print("\nMissing values BEFORE handling:")
print(test_df[FEATURE_COLUMNS].isna().sum()[lambda x: x > 0])

# Handle price availability-dependent features.
# Missing price means price information is unavailable,
# so numerical price inputs are represented as 0 while
# price_available explicitly tells the model that price is unavailable.

price_features = [
    "sell_price",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7"
]

test_df[price_features] = test_df[price_features].fillna(0)

# Initial lag/rolling NaNs are caused by insufficient history.
# Replace them with 0 for the input representation.
history_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28"
]

test_df[history_features] = test_df[history_features].fillna(0)

print("\nMissing values AFTER handling:")
print(test_df[FEATURE_COLUMNS].isna().sum()[lambda x: x > 0])

print("\nFinal input matrix shape:")
print(test_df[FEATURE_COLUMNS].shape)

del test_df
gc.collect()

RNN Input Preparation Test
Rows: 1048576
Features: 22

Missing values BEFORE handling:
sell_price            367871
lag_1                   1000
lag_7                   7000
lag_14                 14000
lag_28                 28000
rolling_mean_7          7000
rolling_mean_28        28000
rolling_std_7           7000
rolling_std_28         28000
price_change_1        367871
price_change_pct_1    367871
price_relative_7      373905
dtype: int64

Missing values AFTER handling:
Series([], dtype: int64)

Final input matrix shape:
(1048576, 22)


9

In [13]:
from sklearn.preprocessing import StandardScaler

# Features that will be scaled
SCALING_COLUMNS = FEATURE_COLUMNS.copy()

# Create scaler
scaler = StandardScaler()

# Training period
TRAIN_END_DATE = pd.Timestamp("2016-03-27")

print("====================================")
print("Fitting RNN Scaler")
print("====================================")

total_train_rows = 0

for rg in range(parquet_file.num_row_groups):

    df = parquet_file.read_row_group(
        rg,
        columns=["date"] + SCALING_COLUMNS
    ).to_pandas()

    df["date"] = pd.to_datetime(df["date"])

    # Keep training period only
    df = df[df["date"] <= TRAIN_END_DATE]

    if len(df) == 0:
        del df
        continue

    # Handle NaNs exactly as tested earlier
    price_features = [
        "sell_price",
        "price_change_1",
        "price_change_pct_1",
        "price_relative_7"
    ]

    history_features = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]

    df[price_features] = df[price_features].fillna(0)
    df[history_features] = df[history_features].fillna(0)

    # Incremental fitting
    scaler.partial_fit(df[SCALING_COLUMNS])

    total_train_rows += len(df)

    del df
    gc.collect()

    if (rg + 1) % 10 == 0:
        print(
            f"Processed row groups: {rg + 1}/{parquet_file.num_row_groups}"
        )

print("\n====================================")
print("Scaler Fitting Complete")
print("====================================")

print("Training rows used:", f"{total_train_rows:,}")
print("Expected training rows:", f"{57_474_000:,}")

print("\nScaler features:", len(scaler.mean_))

print("\nFirst 5 feature means:")
print(
    pd.Series(
        scaler.mean_,
        index=SCALING_COLUMNS
    ).head()
)

print("\nFirst 5 feature standard deviations:")
print(
    pd.Series(
        scaler.scale_,
        index=SCALING_COLUMNS
    ).head()
)

Fitting RNN Scaler
Processed row groups: 10/61
Processed row groups: 20/61
Processed row groups: 30/61
Processed row groups: 40/61
Processed row groups: 50/61
Processed row groups: 60/61

Scaler Fitting Complete
Training rows used: 57,473,650
Expected training rows: 57,474,000

Scaler features: 22

First 5 feature means:
sales               1.122458
sell_price          3.463666
price_available     0.785999
day_of_month       15.714589
week_of_year       26.039788
dtype: float64

First 5 feature standard deviations:
sales               3.876982
sell_price          3.515472
price_available     0.410127
day_of_month        8.793549
week_of_year       15.172256
dtype: float64


In [14]:
# Test the fitted scaler on one training sample

test_df = parquet_file.read_row_group(
    0,
    columns=["date"] + FEATURE_COLUMNS
).to_pandas()

test_df["date"] = pd.to_datetime(test_df["date"])

# Select one training observation
sample = test_df[test_df["date"] <= TRAIN_END_DATE].iloc[[100]].copy()

# Same NaN handling used during scaler fitting
price_features = [
    "sell_price",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7"
]

history_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28"
]

sample[price_features] = sample[price_features].fillna(0)
sample[history_features] = sample[history_features].fillna(0)

# Apply fitted scaler
scaled_sample = scaler.transform(sample[SCALING_COLUMNS])

print("====================================")
print("Scaler Test")
print("====================================")

print("Original shape:", sample[SCALING_COLUMNS].shape)
print("Scaled shape:", scaled_sample.shape)

print("\nAny NaN:", np.isnan(scaled_sample).any())
print("Any infinite:", np.isinf(scaled_sample).any())

print("\nFirst 10 scaled feature values:")
print(scaled_sample[0][:10])

del test_df
del sample
gc.collect()

Scaler Test
Original shape: (1, 22)
Scaled shape: (1, 22)

Any NaN: False
Any infinite: False

First 10 scaled feature values:
[-0.28951861 -0.98526354 -1.91647527 -0.76358125 -0.46399084 -0.47314583
 -0.40831348 -0.63363001 -0.2894102  -0.28867355]


0

In [15]:
# Test actual RNN sequence generation on one item-store series

sequence_columns = [
    "item_id",
    "store_id",
    "date"
] + FEATURE_COLUMNS

# Read first row group
seq_df = parquet_file.read_row_group(
    0,
    columns=sequence_columns
).to_pandas()

seq_df["date"] = pd.to_datetime(seq_df["date"])

# Select the first item-store series
test_item = seq_df["item_id"].iloc[0]
test_store = seq_df["store_id"].iloc[0]

series = (
    seq_df[
        (seq_df["item_id"] == test_item) &
        (seq_df["store_id"] == test_store)
    ]
    .sort_values("date")
    .reset_index(drop=True)
)

# Apply the same NaN handling
price_features = [
    "sell_price",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7"
]

history_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28"
]

series[price_features] = series[price_features].fillna(0)
series[history_features] = series[history_features].fillna(0)

# Convert features to scaled values
X_scaled = scaler.transform(series[FEATURE_COLUMNS])

# Generate sequences
X_sequences = []
y_targets = []

for i in range(SEQUENCE_LENGTH, len(series)):

    X_sequences.append(
        X_scaled[i - SEQUENCE_LENGTH:i]
    )

    y_targets.append(
        series[TARGET_COLUMN].iloc[i]
    )

X_sequences = np.asarray(X_sequences, dtype=np.float32)
y_targets = np.asarray(y_targets, dtype=np.float32)

print("====================================")
print("Actual Sequence Generation Test")
print("====================================")

print("\nItem:", test_item)
print("Store:", test_store)

print("Series length:", len(series))

print("\nNumber of sequences:", len(X_sequences))

print("X shape:", X_sequences.shape)
print("y shape:", y_targets.shape)

print("\nExpected X shape:")
print(f"({len(series) - SEQUENCE_LENGTH}, "
      f"{SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})")

print("\nExpected y shape:")
print(f"({len(series) - SEQUENCE_LENGTH},)")

print("\nAny NaN in X:", np.isnan(X_sequences).any())
print("Any NaN in y:", np.isnan(y_targets).any())

print("\nFirst target:", y_targets[0])
print("First sequence last-day sales:",
      series[TARGET_COLUMN].iloc[SEQUENCE_LENGTH - 1])

del seq_df
del series
del X_scaled
del X_sequences
del y_targets

gc.collect()

Actual Sequence Generation Test

Item: HOBBIES_1_001
Store: CA_1
Series length: 1049

Number of sequences: 1021
X shape: (1021, 28, 22)
y shape: (1021,)

Expected X shape:
(1021, 28, 22)

Expected y shape:
(1021,)

Any NaN in X: False
Any NaN in y: False

First target: 0.0
First sequence last-day sales: 0


0

In [16]:
# Verify the train-validation sequence boundary

TRAIN_END_DATE = pd.Timestamp("2016-03-27")
VALIDATION_START_DATE = pd.Timestamp("2016-03-28")
VALIDATION_END_DATE = pd.Timestamp("2016-04-24")

print("====================================")
print("RNN Train / Validation Split")
print("====================================")

print("\nTraining period:")
print(TRAIN_END_DATE)

print("\nValidation period:")
print(VALIDATION_START_DATE, "→", VALIDATION_END_DATE)

print("\nForecast horizon:", FORECAST_HORIZON)
print("Sequence length:", SEQUENCE_LENGTH)

# First validation prediction:
first_validation_input_end = VALIDATION_START_DATE - pd.Timedelta(days=1)
first_validation_input_start = (
    first_validation_input_end
    - pd.Timedelta(days=SEQUENCE_LENGTH - 1)
)

print("\nFirst validation prediction:")
print("Input start:", first_validation_input_start)
print("Input end:", first_validation_input_end)
print("Target:", VALIDATION_START_DATE)

print("\nInput window length:",
      (first_validation_input_end - first_validation_input_start).days + 1)

# Last validation prediction:
last_validation_input_end = VALIDATION_END_DATE - pd.Timedelta(days=1)
last_validation_input_start = (
    last_validation_input_end
    - pd.Timedelta(days=SEQUENCE_LENGTH - 1)
)

print("\nLast validation prediction:")
print("Input start:", last_validation_input_start)
print("Input end:", last_validation_input_end)
print("Target:", VALIDATION_END_DATE)

print("\nValidation days:",
      (VALIDATION_END_DATE - VALIDATION_START_DATE).days + 1)

RNN Train / Validation Split

Training period:
2016-03-27 00:00:00

Validation period:
2016-03-28 00:00:00 → 2016-04-24 00:00:00

Forecast horizon: 1
Sequence length: 28

First validation prediction:
Input start: 2016-02-29 00:00:00
Input end: 2016-03-27 00:00:00
Target: 2016-03-28 00:00:00

Input window length: 28

Last validation prediction:
Input start: 2016-03-27 00:00:00
Input end: 2016-04-23 00:00:00
Target: 2016-04-24 00:00:00

Validation days: 28


In [17]:
# Test a streaming RNN training dataset
# Only a small number of series are used for this smoke test.

BATCH_SIZE = 64
TEST_SERIES_LIMIT = 100

print("====================================")
print("RNN Streaming Dataset Test")
print("====================================")

print("Batch size:", BATCH_SIZE)
print("Test series limit:", TEST_SERIES_LIMIT)
print("Sequence length:", SEQUENCE_LENGTH)
print("Number of features:", len(FEATURE_COLUMNS))

RNN Streaming Dataset Test
Batch size: 64
Test series limit: 100
Sequence length: 28
Number of features: 22


In [18]:
def prepare_features(df):
    """Apply the same NaN handling used during scaler fitting."""

    price_features = [
        "sell_price",
        "price_change_1",
        "price_change_pct_1",
        "price_relative_7"
    ]

    history_features = [
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]

    df[price_features] = df[price_features].fillna(0)
    df[history_features] = df[history_features].fillna(0)

    return df


def sequence_generator(
    parquet_file,
    scaler,
    feature_columns,
    sequence_length,
    start_date,
    end_date,
    max_series=None
):
    """
    Generate RNN sequences from Parquet without loading
    the complete dataset into memory.
    """

    required_columns = [
        "item_id",
        "store_id",
        "date"
    ] + feature_columns

    series_count = 0
    current_key = None
    current_series = []

    for rg in range(parquet_file.num_row_groups):

        df = parquet_file.read_row_group(
            rg,
            columns=required_columns
        ).to_pandas()

        df["date"] = pd.to_datetime(df["date"])

        # Process rows series-by-series
        for key, group in df.groupby(
            ["item_id", "store_id"],
            sort=False
        ):

            group = group.sort_values("date")

            # Combine with previous rows of the same series
            # in case the series crosses a row-group boundary.
            if current_key == key:
                current_series.append(group)
            else:

                # Finish previous series
                if current_key is not None:

                    full_series = pd.concat(
                        current_series,
                        ignore_index=True
                    ).sort_values("date")

                    # Keep only required date range
                    full_series = full_series[
                        (full_series["date"] >= start_date) &
                        (full_series["date"] <= end_date)
                    ]

                    if len(full_series) > sequence_length:

                        full_series = prepare_features(full_series)

                        X_scaled = scaler.transform(
                            full_series[feature_columns]
                        )

                        for i in range(
                            sequence_length,
                            len(full_series)
                        ):

                            X = X_scaled[
                                i-sequence_length:i
                            ].astype(np.float32)

                            y = np.float32(
                                full_series["sales"].iloc[i]
                            )

                            yield X, y

                # Start new series
                current_key = key
                current_series = [group]

                series_count += 1

                if (
                    max_series is not None
                    and series_count >= max_series
                ):
                    break

        del df
        gc.collect()

        if (
            max_series is not None
            and series_count >= max_series
        ):
            break

In [19]:
# Test the streaming generator on a small number of series

generator = sequence_generator(
    parquet_file=parquet_file,
    scaler=scaler,
    feature_columns=FEATURE_COLUMNS,
    sequence_length=SEQUENCE_LENGTH,
    start_date=pd.Timestamp("2011-01-29"),
    end_date=TRAIN_END_DATE,
    max_series=TEST_SERIES_LIMIT
)

# Get first generated sequence
X_test, y_test = next(generator)

print("====================================")
print("Streaming Generator Test")
print("====================================")

print("X shape:", X_test.shape)
print("y shape:", y_test.shape if hasattr(y_test, "shape") else "scalar")

print("\nExpected X shape:")
print(f"({SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})")

print("\nX dtype:", X_test.dtype)
print("y dtype:", type(y_test))

print("\nAny NaN in X:", np.isnan(X_test).any())
print("Any infinite in X:", np.isinf(X_test).any())
print("y value:", y_test)

Streaming Generator Test
X shape: (28, 22)
y shape: ()

Expected X shape:
(28, 22)

X dtype: float32
y dtype: <class 'numpy.float32'>

Any NaN in X: False
Any infinite in X: False
y value: 0.0


In [20]:
# Create a TensorFlow Dataset from the streaming generator

def make_dataset(
    start_date,
    end_date,
    max_series=None,
    batch_size=BATCH_SIZE
):

    output_signature = (
        tf.TensorSpec(
            shape=(SEQUENCE_LENGTH, len(FEATURE_COLUMNS)),
            dtype=tf.float32
        ),
        tf.TensorSpec(
            shape=(),
            dtype=tf.float32
        )
    )

    dataset = tf.data.Dataset.from_generator(
        lambda: sequence_generator(
            parquet_file=parquet_file,
            scaler=scaler,
            feature_columns=FEATURE_COLUMNS,
            sequence_length=SEQUENCE_LENGTH,
            start_date=pd.Timestamp(start_date),
            end_date=pd.Timestamp(end_date),
            max_series=max_series
        ),
        output_signature=output_signature
    )

    dataset = dataset.batch(batch_size)

    return dataset


# Test dataset using only 100 series
train_dataset_test = make_dataset(
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE,
    max_series=TEST_SERIES_LIMIT,
    batch_size=BATCH_SIZE
)

# Get one batch
X_batch, y_batch = next(iter(train_dataset_test))

print("====================================")
print("TensorFlow Dataset Test")
print("====================================")

print("X batch shape:", X_batch.shape)
print("y batch shape:", y_batch.shape)

print("\nExpected X shape:")
print(f"({BATCH_SIZE}, {SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})")

print("\nExpected y shape:")
print(f"({BATCH_SIZE},)")

print("\nX dtype:", X_batch.dtype)
print("y dtype:", y_batch.dtype)

print("\nAny NaN in X:", tf.reduce_any(tf.math.is_nan(X_batch)).numpy())
print("Any NaN in y:", tf.reduce_any(tf.math.is_nan(y_batch)).numpy())

print("\nAny infinite in X:",
      tf.reduce_any(tf.math.is_inf(X_batch)).numpy())

print("\nFirst target values:")
print(y_batch[:10].numpy())

TensorFlow Dataset Test
X batch shape: (64, 28, 22)
y batch shape: (64,)

Expected X shape:
(64, 28, 22)

Expected y shape:
(64,)

X dtype: <dtype: 'float32'>
y dtype: <dtype: 'float32'>

Any NaN in X: False
Any NaN in y: False

Any infinite in X: False

First target values:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [22]:
# Build the Simple RNN model

model = Sequential([
    tf.keras.layers.Input(
        shape=(SEQUENCE_LENGTH, len(FEATURE_COLUMNS))
    ),

    SimpleRNN(
        64,
        activation="tanh"
    ),

    Dense(
        32,
        activation="relu"
    ),

    Dense(
        1
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=[
        tf.keras.metrics.MeanAbsoluteError(name="mae")
    ]
)

print("====================================")
print("RNN Model Architecture")
print("====================================")

model.summary()

RNN Model Architecture


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         5,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,681 (30.00 KB)

 Trainable params: 7,681 (30.00 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
# RNN training configuration

EPOCHS = 10
BATCH_SIZE = 64

# Number of training sequences per epoch
# We start with a manageable number for model development.
TRAIN_STEPS = 5000

# Number of validation batches evaluated after each epoch
VALIDATION_STEPS = 500

print("====================================")
print("RNN Training Configuration")
print("====================================")

print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Training steps per epoch:", TRAIN_STEPS)
print("Validation steps:", VALIDATION_STEPS)

print("\nTraining period:")
print("2011-01-29 → 2016-03-27")

print("\nValidation period:")
print("2016-03-28 → 2016-04-24")

RNN Training Configuration
Epochs: 10
Batch size: 64
Training steps per epoch: 5000
Validation steps: 500

Training period:
2011-01-29 → 2016-03-27

Validation period:
2016-03-28 → 2016-04-24


In [24]:
# Create the streaming training dataset

train_dataset = make_dataset(
    start_date="2011-01-29",
    end_date=TRAIN_END_DATE,
    max_series=None,
    batch_size=BATCH_SIZE
)

# Shuffle only the generated sequence stream.
# The dataset remains memory-efficient because the shuffle
# buffer contains only a limited number of samples.
train_dataset = train_dataset.shuffle(
    buffer_size=4096,
    reshuffle_each_iteration=True
)

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

print("====================================")
print("Training Dataset Created")
print("====================================")

print("Batch size:", BATCH_SIZE)
print("Training steps per epoch:", TRAIN_STEPS)
print("Shuffle buffer:", 4096)
print("Prefetch: enabled")

Training Dataset Created
Batch size: 64
Training steps per epoch: 5000
Shuffle buffer: 4096
Prefetch: enabled


In [25]:
# Create the streaming validation dataset

validation_dataset = make_dataset(
    start_date=VALIDATION_START_DATE,
    end_date=VALIDATION_END_DATE,
    max_series=None,
    batch_size=BATCH_SIZE
)

validation_dataset = validation_dataset.prefetch(
    tf.data.AUTOTUNE
)

print("====================================")
print("Validation Dataset Created")
print("====================================")

print("Batch size:", BATCH_SIZE)
print("Validation steps:", VALIDATION_STEPS)
print("Validation period:")
print(VALIDATION_START_DATE, "→", VALIDATION_END_DATE)
print("Shuffle: disabled")
print("Prefetch: enabled")

Validation Dataset Created
Batch size: 64
Validation steps: 500
Validation period:
2016-03-28 00:00:00 → 2016-04-24 00:00:00
Shuffle: disabled
Prefetch: enabled


In [26]:
def validation_sequence_generator(
    parquet_file,
    scaler,
    feature_columns,
    sequence_length,
    validation_start,
    validation_end,
    max_series=None
):
    """
    Generate validation sequences for one-step-ahead forecasting.

    Historical observations before validation_start are included
    so that the first validation target has a complete history window.
    """

    required_columns = [
        "item_id",
        "store_id",
        "date"
    ] + feature_columns

    validation_start = pd.Timestamp(validation_start)
    validation_end = pd.Timestamp(validation_end)

    # We need sequence_length days immediately before
    # the first validation target.
    history_start = validation_start - pd.Timedelta(
        days=sequence_length
    )

    series_count = 0
    current_key = None
    current_series = []

    for rg in range(parquet_file.num_row_groups):

        df = parquet_file.read_row_group(
            rg,
            columns=required_columns
        ).to_pandas()

        df["date"] = pd.to_datetime(df["date"])

        # Keep only the history needed before validation
        # plus the complete validation period.
        df = df[
            (df["date"] >= history_start) &
            (df["date"] <= validation_end)
        ]

        for key, group in df.groupby(
            ["item_id", "store_id"],
            sort=False
        ):

            group = group.sort_values("date")

            if current_key == key:
                current_series.append(group)

            else:

                # Process previous series
                if current_key is not None:

                    full_series = pd.concat(
                        current_series,
                        ignore_index=True
                    ).sort_values("date")

                    full_series = prepare_features(full_series)

                    X_scaled = scaler.transform(
                        full_series[feature_columns]
                    )

                    dates = full_series["date"].reset_index(drop=True)

                    for i in range(sequence_length, len(full_series)):

                        target_date = dates.iloc[i]

                        if (
                            target_date >= validation_start
                            and target_date <= validation_end
                        ):

                            X = X_scaled[
                                i-sequence_length:i
                            ].astype(np.float32)

                            y = np.float32(
                                full_series["sales"].iloc[i]
                            )

                            yield X, y

                current_key = key
                current_series = [group]

                series_count += 1

                if (
                    max_series is not None
                    and series_count >= max_series
                ):
                    break

        del df
        gc.collect()

        if (
            max_series is not None
            and series_count >= max_series
        ):
            break

    # Process final series
    if current_key is not None:

        full_series = pd.concat(
            current_series,
            ignore_index=True
        ).sort_values("date")

        full_series = prepare_features(full_series)

        X_scaled = scaler.transform(
            full_series[feature_columns]
        )

        dates = full_series["date"].reset_index(drop=True)

        for i in range(sequence_length, len(full_series)):

            target_date = dates.iloc[i]

            if (
                target_date >= validation_start
                and target_date <= validation_end
            ):

                X = X_scaled[
                    i-sequence_length:i
                ].astype(np.float32)

                y = np.float32(
                    full_series["sales"].iloc[i]
                )

                yield X, y

In [27]:
# Test the corrected validation generator on 100 series

validation_generator = validation_sequence_generator(
    parquet_file=parquet_file,
    scaler=scaler,
    feature_columns=FEATURE_COLUMNS,
    sequence_length=SEQUENCE_LENGTH,
    validation_start=VALIDATION_START_DATE,
    validation_end=VALIDATION_END_DATE,
    max_series=TEST_SERIES_LIMIT
)

# Get the first validation sequence
X_val_test, y_val_test = next(validation_generator)

print("====================================")
print("Validation Generator Test")
print("====================================")

print("X shape:", X_val_test.shape)
print("y shape:", y_val_test.shape if hasattr(y_val_test, "shape") else "scalar")

print("\nExpected X shape:")
print(f"({SEQUENCE_LENGTH}, {len(FEATURE_COLUMNS)})")

print("\nTarget value:", y_val_test)

print("\nAny NaN in X:", np.isnan(X_val_test).any())
print("Any infinite in X:", np.isinf(X_val_test).any())
print("Any NaN in y:", np.isnan(y_val_test).any())

print("\nFirst validation target date should be:")
print(VALIDATION_START_DATE)

Validation Generator Test
X shape: (28, 22)
y shape: ()

Expected X shape:
(28, 22)

Target value: 1.0

Any NaN in X: False
Any infinite in X: False
Any NaN in y: False

First validation target date should be:
2016-03-28 00:00:00


In [28]:
# Create the corrected streaming validation dataset

validation_dataset = tf.data.Dataset.from_generator(
    lambda: validation_sequence_generator(
        parquet_file=parquet_file,
        scaler=scaler,
        feature_columns=FEATURE_COLUMNS,
        sequence_length=SEQUENCE_LENGTH,
        validation_start=VALIDATION_START_DATE,
        validation_end=VALIDATION_END_DATE,
        max_series=None
    ),
    output_signature=(
        tf.TensorSpec(
            shape=(SEQUENCE_LENGTH, len(FEATURE_COLUMNS)),
            dtype=tf.float32
        ),
        tf.TensorSpec(
            shape=(),
            dtype=tf.float32
        )
    )
)

validation_dataset = validation_dataset.batch(BATCH_SIZE)

validation_dataset = validation_dataset.prefetch(
    tf.data.AUTOTUNE
)

print("====================================")
print("Final Validation Dataset")
print("====================================")

print("Batch size:", BATCH_SIZE)
print("Validation period:")
print(VALIDATION_START_DATE, "→", VALIDATION_END_DATE)
print("Shuffle: disabled")
print("Prefetch: enabled")

Final Validation Dataset
Batch size: 64
Validation period:
2016-03-28 00:00:00 → 2016-04-24 00:00:00
Shuffle: disabled
Prefetch: enabled


In [29]:
X_val_batch, y_val_batch = next(iter(validation_dataset))

print("====================================")
print("Validation Batch Test")
print("====================================")

print("X shape:", X_val_batch.shape)
print("y shape:", y_val_batch.shape)

print("\nExpected X shape:")
print(f"(batch_size={BATCH_SIZE}, sequence_length={SEQUENCE_LENGTH}, features={len(FEATURE_COLUMNS)})")

print("\nExpected y shape:")
print(f"(batch_size={BATCH_SIZE},)")

print("\nAny NaN in X:", tf.reduce_any(tf.math.is_nan(X_val_batch)).numpy())
print("Any infinite in X:", tf.reduce_any(tf.math.is_inf(X_val_batch)).numpy())

print("Any NaN in y:", tf.reduce_any(tf.math.is_nan(y_val_batch)).numpy())
print("Any infinite in y:", tf.reduce_any(tf.math.is_inf(y_val_batch)).numpy())

print("\nFirst 10 validation targets:")
print(y_val_batch[:10].numpy())

Validation Batch Test
X shape: (64, 28, 22)
y shape: (64,)

Expected X shape:
(batch_size=64, sequence_length=28, features=22)

Expected y shape:
(batch_size=64,)

Any NaN in X: False
Any infinite in X: False
Any NaN in y: False
Any infinite in y: False

First 10 validation targets:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]


In [30]:
# Train the RNN model

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_dataset,
    steps_per_epoch=TRAIN_STEPS,
    validation_data=validation_dataset,
    validation_steps=VALIDATION_STEPS,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1
)

print("\n====================================")
print("RNN Training Completed")
print("====================================")

print("Epochs completed:", len(history.history["loss"]))
print("Best validation loss:",
      min(history.history["val_loss"]))

Epoch 1/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 269s 33ms/step - loss: 6.6724 - mae: 0.8524 - val_loss: 3.8771 - val_mae: 0.9958
Epoch 2/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 164s 33ms/step - loss: 3.1616 - mae: 0.6583 - val_loss: 3.8629 - val_mae: 0.9878
Epoch 3/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 166s 33ms/step - loss: 2.5707 - mae: 0.6937 - val_loss: 3.9254 - val_mae: 0.9444
Epoch 4/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 168s 34ms/step - loss: 4.2736 - mae: 0.8584 - val_loss: 3.7941 - val_mae: 0.9690
Epoch 5/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 168s 34ms/step - loss: 3.0077 - mae: 0.8509 - val_loss: 3.8043 - val_mae: 1.0072
Epoch 6/10
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 169s 34ms/step - loss: 3.4134 - mae: 0.9341 - val_loss: 3.8737 - val_mae: 0.9911

RNN Training Completed
Epochs completed: 6
Best validation loss: 3.79412841796875


In [31]:
# Generate RNN predictions on the complete validation period

y_true = []
y_pred = []

for X_batch, y_batch in validation_dataset:
    predictions = model.predict(X_batch, verbose=0)

    y_true.append(y_batch.numpy())
    y_pred.append(predictions.squeeze())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

print("====================================")
print("RNN Validation Predictions")
print("====================================")

print("Actual values shape:", y_true.shape)
print("Predicted values shape:", y_pred.shape)

print("\nExpected validation observations:")
print(30490 * 28)

print("\nAny NaN in predictions:", np.isnan(y_pred).any())
print("Any infinite in predictions:", np.isinf(y_pred).any())

print("\nFirst 10 actual values:")
print(y_true[:10])

print("\nFirst 10 predictions:")
print(y_pred[:10])

RNN Validation Predictions
Actual values shape: (853720,)
Predicted values shape: (853720,)

Expected validation observations:
853720

Any NaN in predictions: False
Any infinite in predictions: False

First 10 actual values:
[1. 0. 0. 0. 0. 0. 1. 0. 4. 2.]

First 10 predictions:
[0.9615194  0.96198994 0.8425003  0.71798074 0.7258543  1.0364193
 1.0047252  0.5867876  0.4856629  0.85989493]


In [32]:
# Calculate RNN validation metrics

mae = np.mean(np.abs(y_true - y_pred))

rmse = np.sqrt(
    np.mean((y_true - y_pred) ** 2)
)

wape = (
    np.sum(np.abs(y_true - y_pred))
    / np.sum(np.abs(y_true))
) * 100

print("====================================")
print("RNN Validation Performance")
print("====================================")

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"WAPE : {wape:.4f}%")

RNN Validation Performance
MAE  : 1.0326
RMSE : 2.4927
WAPE : 74.4774%


## 6. RNN Results and Findings

The RNN model was evaluated on the same 28-day time-based validation period used for the baseline models.

### Validation Period

- Training period: 2011-01-29 to 2016-03-27
- Validation period: 2016-03-28 to 2016-04-24
- Validation observations: 853,720
- Item-store series: 30,490

### RNN Performance

| Model | MAE | RMSE | WAPE |
|---|---:|---:|---:|
| Naive | 1.1798 | 2.5948 | 85.10% |
| Seasonal Naive (7-day) | 1.2054 | 2.6601 | 86.94% |
| 28-day Moving Average | **1.0050** | **2.0935** | **72.49%** |
| RNN | 1.0326 | 2.4927 | 74.48% |

### Key Findings

The RNN successfully learned temporal patterns from the historical multivariate observations.

The RNN outperformed both the Naive and Seasonal Naive baselines across the primary evaluation metrics.

However, the RNN did not outperform the 28-day Moving Average baseline.

The 28-day Moving Average achieved the lowest MAE, RMSE, and WAPE among the evaluated models.

The RNN therefore provides a stronger benchmark than the simple one-day and seven-day baselines, but the current RNN configuration does not yet outperform the strongest baseline.

No additional RNN tuning was performed at this stage so that subsequent deep learning models can be evaluated under a consistent experimental framework.

## 7. Conclusion

The RNN forecasting stage has been completed successfully.

The model was trained using 28 days of historical observations and 22 multivariate features to predict next-day demand.

The RNN achieved:

- MAE: **1.0326**
- RMSE: **2.4927**
- WAPE: **74.48%**

Although the RNN improved upon the Naive and Seasonal Naive baselines, it did not outperform the 28-day Moving Average baseline, which achieved a WAPE of **72.49%**.

The RNN results will be retained as a benchmark for comparison with the more advanced sequence models.

The RNN stage is complete.